# IBrary Biology Content Pipeline

Notebook version of `scripts/run_pipeline.py`. Run cells in order.

**Steps:** extract → validate → align → curate → judge → publish

**Prerequisites:** Docker (Postgres + DynamoDB) running, `.env` configured, migrations applied.

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path
from pprint import pprint

In [2]:
# Project root (run notebook from repo root or notebooks/)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data" / "docs" / "extracted_source_content" / "biology"
OUTPUT_DIR = DATA_DIR
TEXTBOOK_PDF = Path(os.environ.get("TEXTBOOK_PDF", "")) if os.environ.get("TEXTBOOK_PDF") else (DATA_DIR / "Biology2e-WEB.pdf")
CURRICULUM_JSON = DATA_DIR / "biology_curriculum_structured.json"

import structlog
structlog.configure(
    processors=[
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.dev.ConsoleRenderer(),
    ],
)
logger = structlog.get_logger("pipeline")

print("Project root:", PROJECT_ROOT)
print("Textbook PDF:", TEXTBOOK_PDF)
print("PDF exists:", TEXTBOOK_PDF.exists())

Project root: c:\Users\Motunrayo Ibiyo\Documents\IBrary
Textbook PDF: c:\Users\Motunrayo Ibiyo\Documents\IBrary\data\docs\extracted_source_content\biology\Biology2e-WEB.pdf
PDF exists: True


## 1. Extract — Textbook PDF → PostgreSQL

In [3]:
from ibrary.config import S3_BUCKET
from ibrary.textbook.openstax_biology2e import extract_openstax_biology_2e
from ibrary.textbook.loader import (
    save_images_locally,
    save_images_to_s3,
    upsert_chunks,
    upsert_textbook,
)

logger.info("step_extract", pdf=str(TEXTBOOK_PDF))
# upsert_textbook(
#     book_id="bio2e",
#     title="Biology 2e (OpenStax)",
#     edition="2e",
#     source_path=str(TEXTBOOK_PDF),
#     subject="biology",
# )

chunks, images = extract_openstax_biology_2e(TEXTBOOK_PDF, book_id="bio2e")


2026-03-22T22:07:47.055146Z [info     ] step_extract                   pdf='c:\\Users\\Motunrayo Ibiyo\\Documents\\IBrary\\data\\docs\\extracted_source_content\\biology\\Biology2e-WEB.pdf'
2026-03-22T22:08:45.054957Z [info     ] extraction_complete            chunks=229 images=1156


In [4]:
type(chunks)

list

In [6]:
type(images)

list

In [8]:
images[0]


ExtractedImage(image_id='bio2e_ch1_sec1_pg28_img0', chunk_id='bio2e_ch1_sec1', page_num=28, ext='jpeg', caption='FIGURE 1.1 This NASA image is a composite of several satellite-based views of Earth. To make the whole-Earth\nimage, NASA scientists combine observations of different parts of the planet. (credit: NASA/GSFC/NOAA/USGS)', alt_text='Figure 1.1: FIGURE 1.1 This NASA image is a composite of several satellite-based views of Earth. To make the whole-Earth\nimage, NASA scientists combine observations of different parts of the planet. (credit: NASA')

In [5]:
pprint(chunks[3])

TextbookChunkRecord(chunk_id='bio2e_ch2_sec1',
                    book_id='bio2e',
                    chapter_num=2,
                    section_num=1,
                    subsection_num=None,
                    title='Atoms, Isotopes, Ions, and Molecules: The Building '
                          'Blocks',
                    summary='At its most fundamental level, life is made up of '
                            'matter. Matter is any substance that occupies\n'
                            'space and has mass. Elements are unique forms of '
                            'matter with specific chemical and physical\n'
                            'properties that cannot break down into smaller '
                            'substances by ordinary chemical reactions. There\n'
                            'are 118 elements, but only 98 occur naturally. '
                            'The remaining elements are unstable and require\n'
                            'scientists to synthesize them

In [8]:
upserted = upsert_chunks(chunks)
logger.info("chunks_loaded", total=len(chunks), upserted=upserted)

# if images:
#     try:
#         save_images_to_s3(images, S3_BUCKET)
#     except Exception:
#         logger.warning("s3_unavailable_saving_locally")
#         save_images_locally(images, OUTPUT_DIR / "textbook_images")

print(f"Chunks: {len(chunks)}, upserted: {upserted}")

2026-03-16T22:10:14.697363Z [info     ] chunks_upserted                count=0
2026-03-16T22:10:14.703352Z [info     ] chunks_loaded                  total=229 upserted=0
Chunks: 229, upserted: 0


## 2. Validate — Curriculum JSON

In [6]:
from ibrary.curriculum.validator import save_validated, validate_curriculum

logger.info("step_validate", json=str(CURRICULUM_JSON))
validated = validate_curriculum(CURRICULUM_JSON)
save_validated(validated, OUTPUT_DIR)
print(f"Validated units: {len(validated.units)}")

2026-03-22T22:33:22.106457Z [info     ] step_validate                  json='c:\\Users\\Motunrayo Ibiyo\\Documents\\IBrary\\data\\docs\\extracted_source_content\\biology\\biology_curriculum_structured.json'
2026-03-22T22:33:22.115517Z [info     ] curriculum_validated           topics=14 units=88 unmapped=27
2026-03-22T22:33:22.127261Z [info     ] curriculum_saved               path='c:\\Users\\Motunrayo Ibiyo\\Documents\\IBrary\\data\\docs\\extracted_source_content\\biology\\curriculum_validated.json'
Validated units: 88


## 3. Align — Embed chunks + align curriculum ↔ textbook

In [ ]:
from ibrary.alignment.aligner import align_all, save_alignment
from ibrary.alignment.embedder import embed_textbook_chunks

logger.info("step_align")
embedded = embed_textbook_chunks()
logger.info("embedding_done", new_embeddings=embedded)

alignment = align_all(validated.units)
save_alignment(alignment, OUTPUT_DIR)
print(f"Alignment entries: {len(alignment)}")

## 4. Curate — Generate UDL content via LLM (RAG)

In [ ]:
from ibrary.curation.curation_service import curate_all, save_curated

logger.info("step_curate")
resume_from = set()
curated_path = OUTPUT_DIR / "curated_content.json"
if curated_path.exists():
    existing = json.loads(curated_path.read_text())
    resume_from = {m["curriculum_unit_id"] for m in existing}
    logger.info("resuming_curation", already_done=len(resume_from))

modules = curate_all(validated.units, alignment, resume_from=resume_from)
save_curated(modules, OUTPUT_DIR)
print(f"Curated modules: {len(modules)}")

In [ ]:
# Textbook similarity for one curriculum subtopic: set TOPIC_NUMBER and CONTENT_TEXT
# (as in curriculum_validated.json). Uses saved alignment + chunk bodies from PostgreSQL.

from sqlalchemy import text

from ibrary.db import get_session

TOPIC_NUMBER = 1
# Exact string from curriculum_validated.json "content_text" for this topic.
CONTENT_TEXT = "Characteristics of living things"
# "exact" — strip-trimmed equality | "substring" — query in content_text (case-insensitive)
CONTENT_TEXT_MATCH = "exact"

_validated_path = OUTPUT_DIR / "curriculum_validated.json"
validated = json.loads(_validated_path.read_text(encoding="utf-8"))
_units = validated.get("units") or []


def _strip(s: object) -> str:
    return (s if isinstance(s, str) else "").strip()


def _pick_units_for_topic_and_content() -> list[dict]:
    q = _strip(CONTENT_TEXT)
    out = []
    for u in _units:
        if u.get("topic_number") != TOPIC_NUMBER:
            continue
        ct = _strip(u.get("content_text"))
        if CONTENT_TEXT_MATCH == "substring":
            if q and q.casefold() in ct.casefold():
                out.append(u)
        elif ct == q:
            out.append(u)
    return out


candidates = _pick_units_for_topic_and_content()
if not candidates:
    avail = sorted(
        {
            _strip(u.get("content_text"))
            for u in _units
            if u.get("topic_number") == TOPIC_NUMBER and _strip(u.get("content_text"))
        }
    )
    msg = (
        f"No unit for topic_number={TOPIC_NUMBER!r} with CONTENT_TEXT_MATCH={CONTENT_TEXT_MATCH!r} "
        f"and query {CONTENT_TEXT!r}.\n"
        f"Sample content_text values for this topic ({len(avail)}):\n"
        + "\n".join(f"  - {a!r}" for a in avail[:25])
    )
    if len(avail) > 25:
        msg += "\n  ..."
    raise ValueError(msg)

unit_row = candidates[0]
if len(candidates) > 1:
    print(f"Note: {len(candidates)} units matched; using {unit_row['curriculum_unit_id']!r}")

unit_id = unit_row["curriculum_unit_id"]

try:
    _alignment = alignment
except NameError:
    _alignment = None
if _alignment is None:
    alignment = json.loads(
        (OUTPUT_DIR / "curriculum_textbook_alignment.json").read_text(encoding="utf-8")
    )


def _matches_for_entry(entry):
    if isinstance(entry, list):
        return entry
    if isinstance(entry, dict):
        return entry.get("matches") or []
    return []


entry = alignment.get(unit_id)
if entry is None:
    raise KeyError(
        f"No alignment for {unit_id!r}. Run the align step (or ensure curriculum_textbook_alignment.json exists)."
    )

matches = [m for m in _matches_for_entry(entry) if isinstance(m, dict)]
chunk_ids = sorted({str(m["chunk_id"]) for m in matches if m.get("chunk_id")})

chunk_text_by_id = {}
session = get_session()
try:
    if chunk_ids:
        placeholders = ", ".join(f":id{i}" for i in range(len(chunk_ids)))
        params = {f"id{i}": cid for i, cid in enumerate(chunk_ids)}
        rows = session.execute(
            text(f"SELECT chunk_id, title, content FROM textbook_chunks WHERE chunk_id IN ({placeholders})"),
            params,
        )
        for row in rows:
            chunk_text_by_id[row.chunk_id] = {"title": row.title, "content": row.content}
finally:
    session.close()

topic_content_textbook_similarity = {
    "curriculum_unit_id": unit_id,
    "topic_number": TOPIC_NUMBER,
    "topic": unit_row.get("topic"),
    "theme_number": unit_row.get("theme_number"),
    "content_text": unit_row.get("content_text"),
    "alignment_matches": matches,
    "textbook_chunks": chunk_text_by_id,
}

print(
    f"Unit {unit_id}: {len(matches)} alignment match(es), "
    f"{len(chunk_text_by_id)} chunk body(ies) from Postgres."
)
pprint(matches)
if chunk_ids:
    cid0 = chunk_ids[0]
    body = chunk_text_by_id.get(cid0, {}).get("content") or ""
    print(f"\nPreview `{cid0}` ({len(body)} chars):\n{body[:1500]}...")


ValueError: No unit for topic_number=4 with CONTENT_TEXT_MATCH='exact' and query 'Characteristics of living things'.
Sample content_text values for this topic (8):
  - 'Diffusion: i. definition of diffusion ii. Process of diffusion iii. Significance of diffusion'
  - 'Effects of agricultural on ecological systems • clearing/burning (I) Bush (ii) tillage effects (iii) fertilization/herbicide their effects (iv) effects of different types of farming on .'
  - 'Oassification of plants: • ·1. (i) Botanical classification shows (e.g. algae, • • spermatophytes) (ii) Agricultural of classification (e.g. fibres, · latex) them (iii) Classification based on cycles (e.g.'
  - 'Osmosis: i. definition of osmosis ii. diffusion of water through a selectively permeable membrane iii. haemolysis iv. plasmolysis v Osmometer of living matetial vi. Biological significance of haemolysis and plasmolysis'
  - 'agricul~ral importance (i) knowledge of pests (types, cycles, control) (ii) diseases (types,'
  - "an

In [8]:
topic_content_textbook_similarity


Chapter 1: 4 curriculum units reference these chunks; 1 chunk bodies loaded from DB.


Optional: inspect the full object in the previous code cell output, or evaluate 	opic_content_textbook_similarity here.


## 5. Judge — UDL evaluation (optional)

In [ ]:
from ibrary.evaluation.udl_judge import evaluate_all, save_evaluation

logger.info("step_judge")
scores, flagged = evaluate_all(modules)
save_evaluation(scores, OUTPUT_DIR)
logger.info("judge_complete", total=len(scores), flagged=len(flagged))
print(f"Scored: {len(scores)}, flagged: {len(flagged)}")

## 6. Publish — Curated content → DynamoDB

In [ ]:
from ibrary.serving.dynamodb_writer import create_table, publish_curated_content

logger.info("step_publish")
create_table()
if curated_path.exists():
    count = publish_curated_content(str(curated_path))
    print(f"Published: {count} items")
else:
    print("No curated_content.json — run curate step first.")

---
**Tip:** Run "Run All" to execute the full pipeline, or run cells one by one to inspect outputs. Load `validated` / `alignment` / `modules` from saved JSON if resuming in a new session.